# SmolVLA × PARC2026 LoRA Fine-tuning（Spatial / Object / Goal 統合）

`lerobot/smolvla_libero_plus`を初期重みとして、LIBERO-plusの
`libero_spatial` / `libero_object` / `libero_goal` 3スイート、
合計27〜30タスクをLoRAで追加学習する。単一モデルでTrack1〜3に対応する方針
（`my_strategy.md`方針1）に基づき、3スイートを混ぜて1つのLoRAを学習する。

このNotebookは**1つの`MODE`スイッチ**で2つの使い方を切り替える。

- `MODE = "holdout_eval"`: 各suiteにつき1基本タスクを学習データから除外
  （3スイート合計3タスク、27タスクで学習）し、学習後にholdoutタスクの
  difficulty L1〜L5バリアント（計15件）だけで汎化性能を検証する。
  **手元での精度検証用。**
- `MODE = "full_submission"`: 全10タスク×3スイート（30タスク）を学習に
  使う。holdout評価は行わない（提出用モデルの学習用）。

holdoutタスク・15件のテストケースの選定根拠は`my_strategy.md`の方針2・2-1、
技術的な調査の詳細は`competition_analysis.md`の「LIBERO-plusの摂動カテゴリ別・
ローカル資産カバレッジ」を参照。摂動カテゴリはBackground TexturesとLight
Conditionsの2つのみ（理由は同セクション参照）。

## 1. Colabランタイムを確認する

Colabのランタイムを GPU へ変更してから実行してください。

In [1]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colabなら/content、それ以外（ローカルGPU機、例: 5090機）なら
# PARC_WORKSPACE_ROOT環境変数（未設定ならホーム直下）を作業ディレクトリにする。
WORKSPACE_ROOT = (
    Path("/content")
    if IN_COLAB
    else Path(
        os.environ.get(
            "PARC_WORKSPACE_ROOT",
            str(Path.home() / "parc_lora_workspace"),
        )
    )
)
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

if not IN_COLAB:
    # setup.sh が既にこのリポジトリのローカル環境向けに解決済みの
    # 「sudo無しでMuJoCo(egl)・Wand(MagickWand)を動かす」仕組み
    # （.local_libs/usr, .local_libs/magick_home）をそのまま再利用する。
    # PARC_REPO_ROOTを指定しなければ、Notebookを開いたカレントディレクトリを
    # リポジトリルートとみなす（リポジトリ直下で起動する想定）。
    def _find_repo_root() -> Path:
        # VS Codeのノートブックカーネルは、cwdをNotebookファイル自身の
        # ディレクトリ（例: <repo>/examples）にすることが多く、リポジトリ
        # ルートとは限らない。".local_libs"を目印に、cwdから上位ディレクトリを
        # 数階層たどって実際のリポジトリルートを探す。
        candidate = Path.cwd().resolve()
        for _ in range(6):
            if (candidate / ".local_libs").is_dir():
                return candidate
            if candidate.parent == candidate:
                break
            candidate = candidate.parent
        return Path.cwd().resolve()

    PARC_REPO_ROOT = Path(
        os.environ.get("PARC_REPO_ROOT", str(_find_repo_root()))
    ).resolve()

    _local_lib_dir = PARC_REPO_ROOT / ".local_libs" / "usr" / "lib" / "x86_64-linux-gnu"
    if _local_lib_dir.is_dir():
        os.environ["LD_LIBRARY_PATH"] = (
            str(_local_lib_dir)
            + os.pathsep
            + os.environ.get("LD_LIBRARY_PATH", "")
        )

    _magick_home = PARC_REPO_ROOT / ".local_libs" / "magick_home"
    if _magick_home.is_dir():
        os.environ["MAGICK_HOME"] = str(_magick_home)

    os.environ.setdefault("MUJOCO_GL", "osmesa")  # robosuite独自のEGLコンテキストがこの機のドライバでは失敗するため

    # 重要: LD_LIBRARY_PATH はここ（起動済みプロセス内）で os.environ に
    # 設定しても、動的リンカにはもう効かない（ld.so はプロセス起動時に一度
    # だけ読むため）。lerobot-train 等を subprocess で起動する分には
    # 上の LD_LIBRARY_PATH 設定で問題ないが、この Notebook プロセス自身が
    # 直接 import する wand・mujoco.osmesa については、依存する共有
    # ライブラリを ctypes で絶対パス指定のまま明示的にプリロードしておく
    # 必要がある（実機で検証済み）。
    if _local_lib_dir.is_dir():
        import ctypes

        for _lib_name in (
            "libfftw3.so.3",
            "liblqr-1.so.0",
            "libraw.so.23",
            "libMagickCore-6.Q16.so.7",
            "libMagickWand-6.Q16.so.7",
            "libOSMesa.so.8",
        ):
            _lib_path = _local_lib_dir / _lib_name
            if _lib_path.is_file():
                ctypes.CDLL(str(_lib_path), mode=ctypes.RTLD_GLOBAL)

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
os.environ["HF_HOME"] = str(WORKSPACE_ROOT / "hf_cache")
os.environ["HF_LEROBOT_HOME"] = str(WORKSPACE_ROOT / "lerobot_cache")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以上が必要です。")

if not torch.cuda.is_available():
    raise RuntimeError("GPUが認識されていません（CUDA版torchが入ったvenvか確認）。")

print(f"IN_COLAB = {IN_COLAB}")
print(f"WORKSPACE_ROOT = {WORKSPACE_ROOT}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

IN_COLAB = False
WORKSPACE_ROOT = /home/s6323004/parc_lora_workspace
GPU: NVIDIA GeForce RTX 5090


/home/s6323004/PARC2026_pre/.local_libs/parc_lora/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


## 2. システムパッケージを準備する

LeRobot、動画デコード、MuJoCoで必要になるパッケージを導入する。

ローカルGPU機（5090機など、sudoが使えない環境）で実行する場合、このセルのapt-getはColabでのみ自動実行される。ほとんどのパッケージは（`libgl1`/`libglib2.0-0`/`libsm6`/`libxext6`/`libexpat1`/`git`/`unzip`を含め）通常すでにシステムに入っている。唯一`ffmpeg`が無いことが多いが、これは学習データの動画デコード用途でしか使わないため、ローカルでは（後述のセルで）`--dataset.video_backend`を`torchcodec`から`pyav`（システムffmpeg不要・wheelに同梱）へ自動的に切り替えることで回避する。MuJoCoのレンダリング（`MUJOCO_GL=egl`）とWand（MagickWand）は、`setup.sh`が`.local_libs/`配下に用意済みのsudo不要な仕組みをそのまま再利用する（`PARC_REPO_ROOT`環境変数でリポジトリの場所を指定できる。未指定ならNotebook起動時のカレントディレクトリを使う）。

In [2]:
def run_quiet(
    command: list[str],
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])

    return result


APT_PACKAGES = [
    "ffmpeg",
    "git",
    "unzip",
    "libgl1",
    "libglib2.0-0",
    "libsm6",
    "libxext6",
    "libexpat1",
    "libfontconfig1-dev",
    "libmagickwand-dev",
]

if IN_COLAB:
    run_quiet(["apt-get", "update", "-qq"])
    run_quiet(["apt-get", "install", "-y", "-qq", *APT_PACKAGES])
    print("System packages ready.")
else:
    print(
        "IN_COLAB=False のため apt-get はスキップした。"
        " git/unzip/libgl1等は通常インストール済み。"
        " ffmpegはvideo_backend=pyavへの切り替えで代替するため不要。"
        " 万一エラーが出た場合のみ、以下を手動で確認すること:\n"
        f"  dpkg -s {' '.join(APT_PACKAGES)}"
    )

IN_COLAB=False のため apt-get はスキップした。 git/unzip/libgl1等は通常インストール済み。 ffmpegはvideo_backend=pyavへの切り替えで代替するため不要。 万一エラーが出た場合のみ、以下を手動で確認すること:
  dpkg -s ffmpeg git unzip libgl1 libglib2.0-0 libsm6 libxext6 libexpat1 libfontconfig1-dev libmagickwand-dev


## 3. LeRobotをインストールする

LeRobot `v0.6.0`を使用する。Colabでの LoRA 学習に必要な互換性調整もこの
セルで適用する。

`av-dep`（PyAV）も併せて入れる。ローカルGPU機など、システムffmpegが無くsudoも使えない環境では、学習データの動画デコードをtorchcodec（システムffmpeg依存）ではなくPyAV（wheelにffmpeg同梱）へ切り替えて使う。

`hydra-core`・`omegaconf`（設定管理）と`mlflow`（実験記録）もあわせて入れる。

In [3]:
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = WORKSPACE_ROOT / "lerobot"
LEROBOT_SRC = LEROBOT_DIR / "src"

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "lerobot",
        "torchao",
    ],
    check=False,
)

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

run_quiet(
    [
        "git",
        "clone",
        "--quiet",
        "--depth",
        "1",
        "--branch",
        LEROBOT_TAG,
        "https://github.com/huggingface/lerobot.git",
        str(LEROBOT_DIR),
    ]
)

smolvlm_source = (
    LEROBOT_SRC
    / "lerobot"
    / "policies"
    / "smolvla"
    / "smolvlm_with_expert.py"
)

if not torch.cuda.is_bf16_supported():
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace(
        'torch_dtype="bfloat16",',
        'torch_dtype="float16",',
        1,
    )
    smolvlm_source.write_text(
        source,
        encoding="utf-8",
    )

train_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_train.py"
)
source = train_script.read_text(encoding="utf-8")
source = source.replace(
    "logging.info(pformat(cfg.to_dict()))",
    "logging.debug(pformat(cfg.to_dict()))",
    1,
)
source = source.replace(
    "disable=inside_slurm(),",
    "disable=True,",
    1,
)
train_script.write_text(
    source,
    encoding="utf-8",
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "-e",
        f"{LEROBOT_DIR}[training,smolvla,peft,av-dep]",
    ]
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "hydra-core",
        "omegaconf",
        "mlflow",
    ]
)

run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=False,
)

for module_name in list(sys.modules):
    if (
        module_name == "lerobot"
        or module_name.startswith("lerobot.")
        or module_name == "torchao"
        or module_name.startswith("torchao.")
    ):
        del sys.modules[module_name]

sys.path = [
    item
    for item in sys.path
    if item not in {
        str(LEROBOT_DIR),
        str(LEROBOT_SRC),
    }
]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchaoの削除に失敗しました。")

import lerobot
import peft

if (
    LEROBOT_SRC.resolve()
    not in Path(lerobot.__file__).resolve().parents
):
    raise RuntimeError("LeRobotの読込先が正しくありません。")

print("LeRobot ready.")

/home/s6323004/PARC2026_pre/.local_libs/parc_lora/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LeRobot ready.


## 4. 学習・評価条件を設定する（Hydra設定ファイルを読み込む）

パラメータは`smolvla_parc_lora_config.yaml`（このNotebookと同じ`examples/`
ディレクトリ）にまとめてある。**値を変えたいときはこのYAMLファイルを編集**
してから、このセル以降を再実行する。

- `mode`: `"holdout_eval"`（3スイートそれぞれ9/10基本タスクで学習し、残り
  1タスク×L1〜L5計15件で汎化性能を評価する）または`"full_submission"`
  （全10基本タスクで学習し、評価はスキップする）
- 1回だけ・一時的にパラメータを変えたい場合は、次のセルの`HYDRA_OVERRIDES`
  に`"training.steps=5000"`のような文字列を追加すればよい（YAMLファイル
  自体は変更しなくてよい）。

あわせて、このセルで**MLflow（ローカルファイル: `WORKSPACE_ROOT/mlruns`）
への記録も開始**する。ハイパーパラメータと、実際に使ったconfigファイルを
記録する。学習loss・holdout評価結果は後段のセルで追記される。
`mlflow ui --backend-store-uri file:<WORKSPACE_ROOT>/mlruns`で
ブラウザから結果を確認できる。

In [ ]:
# --- Hydra: 設定ファイル(smolvla_parc_lora_config.yaml)を読み込む ---
# 値を変えたいときは、このセルではなく同じディレクトリの
# smolvla_parc_lora_config.yaml を編集して、このセル以降を再実行する。
# 1回だけ・一時的に変えたい場合はHYDRA_OVERRIDESに
# "mode=full_submission" のような文字列を追加する
# （configファイル自体は変更しなくてよい）。
import json as _json

import mlflow
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

HYDRA_CONFIG_NAME = "smolvla_parc_lora_config"
HYDRA_OVERRIDES: list[str] = []  # 例: ["mode=full_submission", "training.steps=5000"]


def _find_hydra_config_dir() -> Path:
    candidates = [Path.cwd()]
    if not IN_COLAB and "PARC_REPO_ROOT" in dir():
        candidates.append(PARC_REPO_ROOT / "examples")
    for candidate in candidates:
        if (candidate / f"{HYDRA_CONFIG_NAME}.yaml").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        f"{HYDRA_CONFIG_NAME}.yaml が見つからない。試した場所: {candidates}"
    )


HYDRA_CONFIG_DIR = _find_hydra_config_dir()

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(HYDRA_CONFIG_DIR), version_base=None):
    cfg = compose(config_name=HYDRA_CONFIG_NAME, overrides=HYDRA_OVERRIDES)

MODE = cfg.mode
assert MODE in ("holdout_eval", "full_submission")

BASE_MODEL_REPO = cfg.model.base_model_repo
BASE_MODEL_REVISION = cfg.model.base_model_revision
VLM_REPO = cfg.model.vlm_repo

DATASET_REPO = cfg.dataset.repo_id
DATASET_REVISION = cfg.dataset.revision
TRAIN_EPISODES_PER_TASK = cfg.dataset.train_episodes_per_task

STEPS = cfg.training.steps
LOG_FREQ = cfg.training.log_freq
BATCH_SIZE = cfg.training.batch_size
LEARNING_RATE = cfg.training.learning_rate
FINAL_LEARNING_RATE = cfg.training.final_learning_rate
WARMUP_STEPS = cfg.training.warmup_steps
SEED = cfg.training.seed

LORA_R = cfg.lora.r
LORA_ALPHA = cfg.lora.alpha

EVAL_EPISODES_PER_TASK = cfg.evaluation.episodes_per_task
EVAL_SEED = cfg.evaluation.seed

# 3スイート×10基本タスク・holdoutタスク・holdoutテストケースはconfigファイルの
# ものをそのまま使う（内容はcompe/t1/holdout_test_tasks.csvと同一）。
SUITE_TASK_NAMES = OmegaConf.to_container(cfg.suite_task_names, resolve=True)
HOLDOUT_TASK_NAMES = OmegaConf.to_container(cfg.holdout_task_names, resolve=True)
HOLDOUT_TEST_TASKS = OmegaConf.to_container(cfg.holdout_test_tasks, resolve=True)

EVAL_TASK_IDS = list(range(len(HOLDOUT_TEST_TASKS)))

OUTPUT_DIR = (
    WORKSPACE_ROOT / "outputs" / f"smolvla_parc_lora_{MODE}"
)
MERGED_MODEL_DIR = (
    WORKSPACE_ROOT / f"smolvla_parc_lora_{MODE}_merged"
)
BASELINE_MODEL_DIR = (
    WORKSPACE_ROOT / "smolvla_parc_baseline"
)

BASE_EVAL_DIR = WORKSPACE_ROOT / "eval" / "base"
FINETUNED_EVAL_DIR = WORKSPACE_ROOT / "eval" / MODE
COMPARISON_CSV_PATH = (
    WORKSPACE_ROOT / f"parc_holdout_comparison_{MODE}.csv"
)
MERGED_ZIP_PATH = (
    WORKSPACE_ROOT / f"smolvla_parc_lora_{MODE}_merged.zip"
)

MIXED_PRECISION = (
    "bf16"
    if torch.cuda.is_bf16_supported()
    else "fp16"
)

# ベースモデル（未学習）は毎回同じ入力に対して同じ評価をするだけなので、
# holdout評価結果をWORKSPACE_ROOTにキャッシュし、次回以降は再計算しない。
# ベースモデル・holdoutテストケース・評価パラメータのいずれかが変わったら
# 自動的にキャッシュを無効化する。
BASE_EVAL_CACHE_PATH = WORKSPACE_ROOT / "base_eval_cache.json"


def _base_eval_cache_key() -> dict:
    return {
        "base_model_repo": BASE_MODEL_REPO,
        "base_model_revision": BASE_MODEL_REVISION,
        "vlm_repo": VLM_REPO,
        "holdout_test_tasks": HOLDOUT_TEST_TASKS,
        "eval_episodes_per_task": EVAL_EPISODES_PER_TASK,
        "eval_seed": EVAL_SEED,
    }


def load_cached_base_eval():
    if not BASE_EVAL_CACHE_PATH.is_file():
        return None
    try:
        cached = _json.loads(BASE_EVAL_CACHE_PATH.read_text(encoding="utf-8"))
    except Exception:
        return None
    if cached.get("cache_key") != _base_eval_cache_key():
        return None
    return cached.get("eval_info")


def save_base_eval_cache(eval_info) -> None:
    BASE_EVAL_CACHE_PATH.write_text(
        _json.dumps(
            {"cache_key": _base_eval_cache_key(), "eval_info": eval_info},
            indent=2,
        ),
        encoding="utf-8",
    )


CACHED_BASE_EVAL_INFO = (
    load_cached_base_eval() if MODE == "holdout_eval" else None
)
if CACHED_BASE_EVAL_INFO is not None:
    print("ベースモデルのholdout評価結果はキャッシュ済み（再計算をスキップします）。")

# --- MLflow: ローカルSQLite（WORKSPACE_ROOT/mlflow.db）に記録する ---
# 素の"file:"バックエンドは新しいMLflowではメンテナンスモード（非推奨）に
# なっており、素のfile URIだと例外を投げる。単一マシン内で完結する
# ローカル運用のままSQLiteバックエンドにする（サーバーは不要、
# `mlflow ui --backend-store-uri sqlite:///<db path>`でローカルから見られる）。
# チーム共有のトラッキングサーバーができたら、set_tracking_uriの1行を
# サーバーのURIに変えるだけで切り替えられる。
mlflow.set_tracking_uri(f"sqlite:///{WORKSPACE_ROOT / 'mlflow.db'}")
mlflow.set_experiment(cfg.mlflow.experiment_name)

if mlflow.active_run() is not None:
    mlflow.end_run()

MLFLOW_RUN = mlflow.start_run(
    run_name=f"{MODE}_steps{STEPS}_r{LORA_R}_alpha{LORA_ALPHA}"
)
mlflow.log_params(
    {
        "mode": MODE,
        "base_model_repo": BASE_MODEL_REPO,
        "base_model_revision": BASE_MODEL_REVISION,
        "vlm_repo": VLM_REPO,
        "dataset_repo": DATASET_REPO,
        "dataset_revision": DATASET_REVISION,
        "train_episodes_per_task": TRAIN_EPISODES_PER_TASK,
        "steps": STEPS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "final_learning_rate": FINAL_LEARNING_RATE,
        "warmup_steps": WARMUP_STEPS,
        "seed": SEED,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "eval_episodes_per_task": EVAL_EPISODES_PER_TASK,
        "eval_seed": EVAL_SEED,
    }
)
# 使ったconfigファイルそのものをartifactとして残す（holdoutタスク一覧等、
# paramsに載せていない詳細もこれで追跡できる）。
mlflow.log_artifact(str(HYDRA_CONFIG_DIR / f"{HYDRA_CONFIG_NAME}.yaml"))

print(f"MODE = {MODE}")
print(f"MLflow run: {MLFLOW_RUN.info.run_id} (tracking_uri={mlflow.get_tracking_uri()})")

# --- 汎用: 任意のメトリクスを「生値」+「移動平均」の両方でMLflowに記録する ---
# 学習lossだけでなく、評価側で新たに記録する指標（成功率・衝突率等）にも
# 同じ関数をそのまま使い回せるようにしている。
from collections import defaultdict, deque

MOVING_AVERAGE_WINDOW = 20

_ma_history: dict[str, deque] = defaultdict(
    lambda: deque(maxlen=MOVING_AVERAGE_WINDOW)
)


def log_metric_ma(
    name: str,
    value: float,
    step: int | None = None,
    window: int = MOVING_AVERAGE_WINDOW,
) -> None:
    """rawの値と、直近window件の移動平均の両方をMLflowに記録する。"""
    mlflow.log_metric(name, value, step=step)
    hist = _ma_history[name]
    if hist.maxlen != window:
        hist = deque(hist, maxlen=window)
        _ma_history[name] = hist
    hist.append(value)
    mlflow.log_metric(f"{name}_ma{window}", sum(hist) / len(hist), step=step)


def log_metrics_ma(
    metrics: dict[str, float],
    step: int | None = None,
    window: int = MOVING_AVERAGE_WINDOW,
) -> None:
    for name, value in metrics.items():
        log_metric_ma(name, value, step=step, window=window)

MODE = holdout_eval


## 5. 公開ファイルの取得処理を用意する

キャッシュを優先し、匿名アクセスの制限時は自動的に再試行する。

In [5]:
import random
import time
from collections.abc import Callable
from typing import TypeVar

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.errors import (
    HfHubHTTPError,
    LocalEntryNotFoundError,
)

T = TypeVar("T")


def run_hf_with_retry(
    operation: Callable[[], T],
) -> T:
    last_error: BaseException | None = None

    for attempt in range(6):
        try:
            return operation()
        except (
            HfHubHTTPError,
            httpx.HTTPStatusError,
        ) as error:
            last_error = error
            response = getattr(error, "response", None)
            status = getattr(response, "status_code", None)

            if status != 429 and "429" not in str(error):
                raise

            if attempt == 5:
                break

            headers = getattr(response, "headers", {}) or {}
            try:
                delay = float(
                    headers.get("Retry-After", 15)
                ) + 1
            except (TypeError, ValueError):
                delay = min(
                    120,
                    15 * (2**attempt) + random.random(),
                )

            time.sleep(delay)

    raise RuntimeError(
        "Hugging Faceからの取得に失敗しました。"
    ) from last_error


def cached_or_downloaded_snapshot(
    repo_id: str,
    revision: str,
    *,
    allow_patterns: list[str] | None = None,
    ignore_patterns: list[str] | None = None,
) -> Path:
    try:
        return Path(
            snapshot_download(
                repo_id=repo_id,
                revision=revision,
                token=False,
                allow_patterns=allow_patterns,
                ignore_patterns=ignore_patterns,
                local_files_only=True,
            )
        )
    except (
        LocalEntryNotFoundError,
        FileNotFoundError,
    ):
        return Path(
            run_hf_with_retry(
                lambda: snapshot_download(
                    repo_id=repo_id,
                    revision=revision,
                    token=False,
                    allow_patterns=allow_patterns,
                    ignore_patterns=ignore_patterns,
                    max_workers=1,
                )
            )
        )

## 6. 3スイート分の学習データを選ぶ

`MODE`に応じて、各suiteの学習対象タスクを決める（`holdout_eval`なら
9/10タスク、`full_submission`なら10/10タスク）。対象タスクそれぞれから
`TRAIN_EPISODES_PER_TASK`件を等間隔に選択し、3スイート分を1つの
`EPISODE_INDICES`にまとめて、後段で単一のLoRA学習にかける。

In [6]:
import re
from collections import defaultdict

from lerobot.datasets.dataset_metadata import (
    LeRobotDatasetMetadata,
)


def normalize_task_name(value: str) -> str:
    value = value.lower().replace("_", " ")
    value = re.sub(r"[^a-z0-9 ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def task_name_from_cell(value) -> str:
    if isinstance(value, str):
        return value

    try:
        if len(value) > 0:
            return str(value[0])
    except TypeError:
        pass

    return str(value)


def choose_evenly_spaced(
    episode_indices: list[int],
    count: int,
) -> list[int]:
    positions = [
        round(
            index
            * (len(episode_indices) - 1)
            / (count - 1)
        )
        for index in range(count)
    ]

    return [
        episode_indices[position]
        for position in positions
    ]


dataset_metadata = run_hf_with_retry(
    lambda: LeRobotDatasetMetadata(
        DATASET_REPO,
        revision=DATASET_REVISION,
    )
)

task_to_episodes: dict[str, list[int]] = defaultdict(list)

for episode_index, task_cell in enumerate(
    dataset_metadata.episodes["tasks"]
):
    task_to_episodes[
        task_name_from_cell(task_cell)
    ].append(int(episode_index))

available_by_normalized = {
    normalize_task_name(task_name): task_name
    for task_name in task_to_episodes
}

EPISODE_INDICES: list[int] = []
TRAIN_TASK_SUMMARY: dict[str, list[str]] = {}

for suite, task_names in SUITE_TASK_NAMES.items():
    holdout_name = HOLDOUT_TASK_NAMES[suite]

    if holdout_name not in task_names:
        raise RuntimeError(
            f"HOLDOUT_TASK_NAMES[{suite!r}] not found in SUITE_TASK_NAMES[{suite!r}]"
        )

    active_names = (
        task_names
        if MODE == "full_submission"
        else [name for name in task_names if name != holdout_name]
    )

    TRAIN_TASK_SUMMARY[suite] = active_names

    for task_name in active_names:
        actual_task = available_by_normalized.get(
            normalize_task_name(task_name)
        )

        if actual_task is None:
            raise RuntimeError(
                f"Task not found in dataset ({suite}): {task_name}"
            )

        EPISODE_INDICES.extend(
            choose_evenly_spaced(
                task_to_episodes[actual_task],
                TRAIN_EPISODES_PER_TASK,
            )
        )

EPISODE_INDICES = sorted(set(EPISODE_INDICES))

expected_tasks = sum(
    len(names) for names in TRAIN_TASK_SUMMARY.values()
)
expected_episodes = expected_tasks * TRAIN_EPISODES_PER_TASK

if len(EPISODE_INDICES) != expected_episodes:
    raise RuntimeError(
        "Episode selection failed: expected "
        f"{expected_episodes}, got {len(EPISODE_INDICES)}"
    )

for suite, names in TRAIN_TASK_SUMMARY.items():
    print(
        f"{suite}: {len(names)} tasks x "
        f"{TRAIN_EPISODES_PER_TASK} episodes"
        + ("  (holdout excluded)" if MODE == "holdout_eval" else "")
    )

print(
    f"Total training data: {expected_tasks} tasks x "
    f"{TRAIN_EPISODES_PER_TASK} episodes = "
    f"{len(EPISODE_INDICES)} episodes"
)

libero_spatial: 9 tasks x 5 episodes  (holdout excluded)
libero_object: 9 tasks x 5 episodes  (holdout excluded)
libero_goal: 9 tasks x 5 episodes  (holdout excluded)
Total training data: 27 tasks x 5 episodes = 135 episodes


/home/s6323004/PARC2026_pre/.local_libs/parc_lora/venv/lib/python3.12/site-packages/datasets/utils/tqdm.py:86: UserWarning: Cannot enable progress bars: environment variable `HF_DATASETS_DISABLE_PROGRESS_BARS=1` is set and has priority.
  warnings.warn(


## 7. 初期重みを準備する

In [7]:
BASE_MODEL_LOCAL = cached_or_downloaded_snapshot(
    BASE_MODEL_REPO,
    BASE_MODEL_REVISION,
    allow_patterns=[
        "config.json",
        "model.safetensors",
        "train_config.json",
        "policy_preprocessor.json",
        "policy_preprocessor*.safetensors",
        "policy_postprocessor.json",
        "policy_postprocessor*.safetensors",
    ],
    ignore_patterns=[
        "README.md",
        "eval/**",
    ],
)

if not (
    BASE_MODEL_LOCAL / "model.safetensors"
).is_file():
    raise FileNotFoundError("Base model not found.")

print("Base model ready.")

Base model ready.


## 8. LoRA学習を実行する

3スイート合計（`holdout_eval`なら27タスク、`full_submission`なら30タスク）
分のエピソードをまとめて1つのLoRAとして学習する。100 stepごとに平均loss
とlearning rateを表示する。

In [8]:
import re
from collections import deque

episodes_json = (
    "["
    + ",".join(map(str, EPISODE_INDICES))
    + "]"
)

command = [
    "lerobot-train",
    f"--policy.path={BASE_MODEL_LOCAL}",
    f"--policy.vlm_model_name={VLM_REPO}",
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    "--policy.input_features=null",
    "--policy.output_features=null",
    "--policy.empty_cameras=0",
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    f"--policy.optimizer_lr={LEARNING_RATE}",
    f"--policy.scheduler_decay_lr={FINAL_LEARNING_RATE}",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}",
    f"--policy.scheduler_decay_steps={STEPS}",
    f"--dataset.repo_id={DATASET_REPO}",
    f"--dataset.revision={DATASET_REVISION}",
    f"--dataset.episodes={episodes_json}",
    "--dataset.use_imagenet_stats=false",
    (
        "--dataset.video_backend="
        + ("torchcodec" if IN_COLAB else "pyav")
    ),
    f"--output_dir={OUTPUT_DIR}",
    f"--job_name=smolvla_parc_lora_{MODE}",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--num_workers=8",
    "--persistent_workers=true",
    "--env_eval_freq=0",
    "--eval_steps=0",
    f"--seed={SEED}",
    "--save_checkpoint=true",
    f"--save_freq={STEPS}",
    "--save_checkpoint_to_hub=false",
    f"--log_freq={LOG_FREQ}",
    "--wandb.enable=false",
    "--peft.method_type=LORA",
    f"--peft.r={LORA_R}",
    f"--peft.lora_alpha={LORA_ALPHA}",
]

training_env = os.environ.copy()
training_env["PYTHONPATH"] = (
    str(LEROBOT_SRC)
    + os.pathsep
    + training_env.get("PYTHONPATH", "")
)
training_env["ACCELERATE_MIXED_PRECISION"] = (
    MIXED_PRECISION
)
training_env["PYTHONUNBUFFERED"] = "1"
training_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_HUB_VERBOSITY"] = "error"
training_env["TQDM_DISABLE"] = "1"
training_env["PYTHONWARNINGS"] = "ignore"

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

print("Preparing data and starting training...")

process = subprocess.Popen(
    command,
    cwd=LEROBOT_DIR,
    env=training_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

recent_lines: deque[str] = deque(maxlen=80)
report_step = LOG_FREQ

assert process.stdout is not None

for raw_line in process.stdout:
    line = raw_line.replace("\r", "").strip()

    if not line:
        continue

    recent_lines.append(line)

    if "step:" in line and "loss:" in line:
        loss_match = re.search(
            r"loss:([0-9.eE+-]+)",
            line,
        )
        lr_match = re.search(
            r"lr:([0-9.eE+-]+)",
            line,
        )

        loss = (
            loss_match.group(1)
            if loss_match
            else "n/a"
        )
        lr = (
            lr_match.group(1)
            if lr_match
            else "n/a"
        )

        print(
            f"step {report_step:4d}/{STEPS}  "
            f"loss={loss}  lr={lr}"
        )
        mlflow_metrics = {}
        if loss_match:
            mlflow_metrics["train_loss"] = float(loss)
        if lr_match:
            mlflow_metrics["train_lr"] = float(lr)
        if mlflow_metrics:
            log_metrics_ma(mlflow_metrics, step=report_step)
        report_step += LOG_FREQ

return_code = process.wait()

if return_code != 0:
    print("\n".join(recent_lines))
    raise RuntimeError(
        f"Training failed: {return_code}"
    )

print("Training complete.")

Preparing data and starting training...
step  100/3000  loss=0.095  lr=1.5e-04
step  200/3000  loss=0.119  lr=3.0e-04
step  300/3000  loss=0.142  lr=3.0e-04
step  400/3000  loss=0.146  lr=2.9e-04
step  500/3000  loss=0.116  lr=2.9e-04
step  600/3000  loss=0.106  lr=2.8e-04
step  700/3000  loss=0.108  lr=2.7e-04
step  800/3000  loss=0.141  lr=2.6e-04
step  900/3000  loss=0.119  lr=2.5e-04
step 1000/3000  loss=0.106  lr=2.4e-04
step 1100/3000  loss=0.121  lr=2.3e-04
step 1200/3000  loss=0.127  lr=2.1e-04
step 1300/3000  loss=0.117  lr=2.0e-04
step 1400/3000  loss=0.099  lr=1.9e-04
step 1500/3000  loss=0.096  lr=1.7e-04
step 1600/3000  loss=0.111  lr=1.6e-04
step 1700/3000  loss=0.126  lr=1.4e-04
step 1800/3000  loss=0.100  lr=1.3e-04
step 1900/3000  loss=0.117  lr=1.2e-04
step 2000/3000  loss=0.127  lr=1.0e-04
step 2100/3000  loss=0.109  lr=9.1e-05
step 2200/3000  loss=0.118  lr=8.0e-05
step 2300/3000  loss=0.094  lr=7.0e-05
step 2400/3000  loss=0.120  lr=6.0e-05
step 2500/3000  loss=0.0

## 9. LoRAをマージしてモデル全体を保存する

LoRA差分を元weightへ統合し、通常のLeRobotモデルとして保存する。

In [9]:
import contextlib
import gc
import io
import json

from peft import PeftModel
from safetensors import safe_open
from lerobot.configs import PreTrainedConfig
from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy,
)

checkpoint_dir = (
    OUTPUT_DIR
    / "checkpoints"
    / f"{STEPS:06d}"
    / "pretrained_model"
)

if not (
    checkpoint_dir / "adapter_model.safetensors"
).is_file():
    raise FileNotFoundError("Final adapter not found.")

gc.collect()
torch.cuda.empty_cache()

merge_config = PreTrainedConfig.from_pretrained(
    checkpoint_dir
)
merge_config.device = "cpu"
merge_config.pretrained_path = BASE_MODEL_LOCAL
merge_config.use_peft = False

quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    base_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=merge_config,
        strict=False,
    )

    peft_policy = PeftModel.from_pretrained(
        base_policy,
        checkpoint_dir,
        is_trainable=False,
        torch_device="cpu",
    )

    merged_policy = peft_policy.merge_and_unload(
        safe_merge=True
    )

shutil.rmtree(MERGED_MODEL_DIR, ignore_errors=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

merged_policy.config.use_peft = False
merged_policy.config.pretrained_path = None
merged_policy.config.push_to_hub = False
merged_policy.config.repo_id = None
merged_policy.config.device = None
merged_policy.config.load_vlm_weights = False
merged_policy.config.vlm_model_name = VLM_REPO

merged_policy.save_pretrained(MERGED_MODEL_DIR)

for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in checkpoint_dir.glob(pattern):
        shutil.copy2(
            source_path,
            MERGED_MODEL_DIR / source_path.name,
        )

merged_weights_path = (
    MERGED_MODEL_DIR / "model.safetensors"
)

with safe_open(
    merged_weights_path,
    framework="pt",
    device="cpu",
) as weights:
    if any(
        "lora_" in key.lower()
        for key in weights.keys()
    ):
        raise RuntimeError(
            "LoRA parameters remain after merge."
        )

del peft_policy
del base_policy
del merged_policy

gc.collect()
torch.cuda.empty_cache()

print("Merged model ready.")

Merged model ready.


## 10. 比較用ベースラインを準備する（`holdout_eval`モードのみ）

公開weightを追加学習モデルと同じ入力schema・processorへ揃える。
`full_submission`モードでは比較評価を行わないためスキップする。

In [10]:
if MODE == "holdout_eval":
    if CACHED_BASE_EVAL_INFO is not None:
        print(
            "ベースモデルのholdout評価はキャッシュ済みのため、"
            "ベースラインモデルの準備をスキップします。"
        )
    else:
        baseline_config = PreTrainedConfig.from_pretrained(
            MERGED_MODEL_DIR
        )
        baseline_config.device = "cpu"
        baseline_config.pretrained_path = BASE_MODEL_LOCAL
        baseline_config.use_peft = False
        baseline_config.load_vlm_weights = False

        quiet_output = io.StringIO()

        with (
            contextlib.redirect_stdout(quiet_output),
            contextlib.redirect_stderr(quiet_output),
        ):
            baseline_policy = SmolVLAPolicy.from_pretrained(
                BASE_MODEL_LOCAL,
                config=baseline_config,
                strict=False,
            )

        shutil.rmtree(BASELINE_MODEL_DIR, ignore_errors=True)
        BASELINE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

        baseline_policy.config.use_peft = False
        baseline_policy.config.pretrained_path = None
        baseline_policy.config.push_to_hub = False
        baseline_policy.config.repo_id = None
        baseline_policy.config.device = None
        baseline_policy.config.load_vlm_weights = False
        baseline_policy.config.vlm_model_name = VLM_REPO
        baseline_policy.save_pretrained(BASELINE_MODEL_DIR)

        for pattern in [
            "policy_preprocessor.json",
            "policy_preprocessor*.safetensors",
            "policy_postprocessor.json",
            "policy_postprocessor*.safetensors",
        ]:
            for source_path in MERGED_MODEL_DIR.glob(pattern):
                shutil.copy2(
                    source_path,
                    BASELINE_MODEL_DIR / source_path.name,
                )

        del baseline_policy
        gc.collect()
        torch.cuda.empty_cache()

        print("Baseline ready.")
else:
    print('Skipped baseline preparation (MODE=full_submission).')


Baseline ready.


## 11. LIBERO-plus評価環境を準備する（`holdout_eval`モードのみ）

MuJoCo、LIBERO-plus fork、評価assetsを導入し、あわせて
`compe/t1/holdout_test_tasks.csv`と同内容（`HOLDOUT_TEST_TASKS`）を
`libero_holdout`という新しいベンチマークスイートとして登録する
（`compe/t1/register.py`の`register_t1`と同じパターン。ただし
`register.py`自体は変更していない — 理由は`my_strategy.md`方針2-1参照）。
`full_submission`モードでは評価を行わないため、この重いセットアップ自体を
スキップする。

登録したスイートは、次のセクションで`pipeline`（この競技専用の評価
ハーネス）から参照する。

In [11]:
if MODE == "holdout_eval":
    from huggingface_hub import hf_hub_download

    LIBERO_PLUS_SHA = "4976dc3"
    LIBERO_PLUS_DIR = WORKSPACE_ROOT / "LIBERO-plus"
    LIBERO_PLUS_PACKAGE_ROOT = (
        LIBERO_PLUS_DIR / "libero" / "libero"
    )
    LIBERO_PLUS_ASSETS_DIR = (
        LIBERO_PLUS_PACKAGE_ROOT / "assets"
    )

    # setdefaultにする: セル2でローカル実行時はosmesaを既に設定済みなので
    # 上書きしない（Colabではここで初めてeglが設定される）。
    os.environ.setdefault("MUJOCO_GL", "egl")

    run_quiet(
        [
            sys.executable,
            "-m",
            "pip",
            "uninstall",
            "-y",
            "hf-libero",
            "libero",
            "robosuite",
        ],
        check=False,
    )

    # robosuiteは--no-depsで入れる。フルインストールすると依存の pynput が
    # evdevをビルドしようとし、sudoが使えない環境ではPython.hが無くビルドに
    # 失敗する（pynput/evdevは人手によるteleoperation入力デバイス用で、
    # ヘッドレスな学習・評価では使わない）。robosuite本体が実際に使う依存
    # （numpy/numba/scipy/Pillow/opencv-python/termcolor）は下のコマンドで
    # 個別に入れる。
    run_quiet(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-deps",
            "robosuite==1.4.1",
        ]
    )

    run_quiet(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "numpy>=1.13.3",
            "numba>=0.49.1",
            "scipy>=1.2.3",
            "Pillow",
            "opencv-python",
            "termcolor",
            "bddl==1.0.1",
            "easydict==1.13",
            "mujoco==3.7.0",
            "matplotlib==3.10.8",
            "Wand==0.6.13",
            "scikit-image==0.25.2",
            "gym==0.26.2",
            # bddl(LIBERO-plus)がPython2/3互換用に使うfuture.utils.with_metaclass。
            # Colabにはあらかじめ入っているためこのリストに無かったが、
            # ローカルの新規venvには入っていないので明示的に追加する。
            "future",
        ]
    )

    if (
        importlib.metadata.version("robosuite")
        != "1.4.1"
    ):
        raise RuntimeError(
            "robosuite 1.4.1 is required."
        )

    if not (LIBERO_PLUS_DIR / ".git").is_dir():
        shutil.rmtree(
            LIBERO_PLUS_DIR,
            ignore_errors=True,
        )
        run_quiet(
            [
                "git",
                "clone",
                "--quiet",
                "https://github.com/sylvestf/LIBERO-plus.git",
                str(LIBERO_PLUS_DIR),
            ]
        )

    checkout = run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "checkout",
            "--quiet",
            LIBERO_PLUS_SHA,
        ],
        check=False,
    )

    if checkout.returncode != 0:
        run_quiet(
            [
                "git",
                "-C",
                str(LIBERO_PLUS_DIR),
                "fetch",
                "--quiet",
                "--depth",
                "1",
                "origin",
                LIBERO_PLUS_SHA,
            ]
        )
        run_quiet(
            [
                "git",
                "-C",
                str(LIBERO_PLUS_DIR),
                "checkout",
                "--quiet",
                LIBERO_PLUS_SHA,
            ]
        )

    run_quiet(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-deps",
            "-e",
            str(LIBERO_PLUS_DIR),
        ]
    )

    if not LIBERO_PLUS_ASSETS_DIR.is_dir():
        assets_root = (
            WORKSPACE_ROOT / "libero_plus_assets"
        )
        archive_path = Path(
            run_hf_with_retry(
                lambda: hf_hub_download(
                    repo_id="Sylvest/LIBERO-plus",
                    repo_type="dataset",
                    filename="assets.zip",
                    local_dir=assets_root,
                    token=False,
                )
            )
        )
        extract_dir = assets_root / "extract"

        shutil.rmtree(extract_dir, ignore_errors=True)
        extract_dir.mkdir(parents=True, exist_ok=True)

        run_quiet(
            [
                "unzip",
                "-q",
                str(archive_path),
                "-d",
                str(extract_dir),
            ]
        )

        candidates = sorted(
            [
                path
                for path in extract_dir.rglob("assets")
                if path.is_dir()
            ],
            key=lambda path: len(path.parts),
        )

        if not candidates:
            raise FileNotFoundError(
                "LIBERO-plus assets not found."
            )

        LIBERO_PLUS_ASSETS_DIR.parent.mkdir(
            parents=True,
            exist_ok=True,
        )
        shutil.move(
            str(candidates[0]),
            str(LIBERO_PLUS_ASSETS_DIR),
        )
        shutil.rmtree(assets_root, ignore_errors=True)

    libero_config_dir = Path.home() / ".libero"
    libero_config_dir.mkdir(
        parents=True,
        exist_ok=True,
    )
    (libero_config_dir / "config.yaml").write_text(
        "\n".join(
            [
                f"assets: {LIBERO_PLUS_ASSETS_DIR}",
                (
                    "bddl_files: "
                    f"{LIBERO_PLUS_PACKAGE_ROOT / 'bddl_files'}"
                ),
                (
                    "datasets: "
                    f"{LIBERO_PLUS_PACKAGE_ROOT.parent / 'datasets'}"
                ),
                (
                    "init_states: "
                    f"{LIBERO_PLUS_PACKAGE_ROOT / 'init_files'}"
                ),
            ]
        )
        + "\n",
        encoding="utf-8",
    )

    eval_script = (
        LEROBOT_SRC
        / "lerobot"
        / "scripts"
        / "lerobot_eval.py"
    )
    source = eval_script.read_text(encoding="utf-8")

    source = source.replace(
        "logging.info(pformat(asdict(cfg)))",
        "logging.debug(pformat(asdict(cfg)))",
        1,
    )
    source = source.replace(
        "max_episodes_rendered = 0 if cfg.eval.recording else 10",
        "max_episodes_rendered = 0",
        1,
    )
    source = source.replace(
        "disable=inside_slurm()",
        "disable=True",
    )

    progress_state = (
        '_EVAL_PROGRESS = {"task_index": 0, "task_total": 0}'
    )
    if progress_state not in source:
        import_anchor = "from tqdm import trange\n"
        if import_anchor not in source:
            raise RuntimeError(
                "Evaluation progress import anchor not found."
            )
        source = source.replace(
            import_anchor,
            import_anchor + "\n" + progress_state + "\n",
            1,
        )

    # lerobot-eval は独立したsubprocessとして起動されるため、Notebook側の
    # 変数（HOLDOUT_TEST_TASKS）や、この場で定義したregister_holdout相当の
    # 関数はそのままでは届かない。そのため、"libero_holdout"スイートの登録
    # コードをlerobot_eval.py自体のソースに直接埋め込み、subprocess起動時に
    # そのプロセス内で確実に登録されるようにする
    # （compe/t1/register.pyのregister_t1と同じ登録パターン。
    # pipeline/environment.pyがregister_t1をin-processで呼ぶのと同じ理由）。
    holdout_marker = "_LIBERO_HOLDOUT_REGISTERED = True"
    if holdout_marker not in source:
        import_anchor = "from tqdm import trange\n"
        if import_anchor not in source:
            raise RuntimeError(
                "Evaluation progress import anchor not found."
            )
        holdout_registration_code = f'''
import re as _holdout_re
import torch as _holdout_torch
from libero.libero import benchmark as _holdout_benchmark
from libero.libero import get_libero_path as _holdout_get_libero_path

_HOLDOUT_TEST_TASKS = {HOLDOUT_TEST_TASKS!r}
_HOLDOUT_SUFFIX_RE = _holdout_re.compile(r"_light_[^.]*|_table_\\d+")


def _holdout_init_states_for(task):
    fn = task.init_states_file
    root = _holdout_get_libero_path("init_states")
    stem, ext = __import__("os").path.splitext(fn)
    stripped = _HOLDOUT_SUFFIX_RE.sub("", stem) + ext
    p = __import__("os").path.join(root, task.problem_folder, stripped)
    return _holdout_torch.load(p, weights_only=False)


_holdout_benchmark.task_maps["libero_holdout"] = {{
    row["task_id"]: _holdout_benchmark.Task(
        name=row["task_id"],
        language=row["instruction"],
        problem="Libero",
        problem_folder=row["suite"],
        bddl_file=f"{{row[\'task_id\']}}.bddl",
        init_states_file=f"{{row[\'task_id\']}}.pruned_init",
    )
    for row in _HOLDOUT_TEST_TASKS
}}
if "libero_holdout" not in _holdout_benchmark.libero_suites:
    _holdout_benchmark.libero_suites.append("libero_holdout")


@_holdout_benchmark.register_benchmark
class LIBERO_HOLDOUT(_holdout_benchmark.Benchmark):
    def __init__(self, task_order_index: int = 0):
        assert task_order_index == 0
        super().__init__(task_order_index=task_order_index)
        self.name = "libero_holdout"
        self.tasks = list(_holdout_benchmark.task_maps[self.name].values())
        self.n_tasks = len(self.tasks)

    def get_task_init_states(self, i):
        return _holdout_init_states_for(self.tasks[i])


{holdout_marker}
'''
        source = source.replace(
            import_anchor,
            import_anchor + holdout_registration_code,
            1,
        )

    task_loop_anchor = (
        "        for i, (task_group, task_id, env) "
        "in enumerate(tasks):\n"
    )
    task_loop_patch = (
        task_loop_anchor
        + '            _EVAL_PROGRESS["task_index"] = i + 1\n'
        + '            _EVAL_PROGRESS["task_total"] = len(tasks)\n'
    )
    if (
        '_EVAL_PROGRESS["task_index"] = i + 1'
        not in source
    ):
        if task_loop_anchor not in source:
            raise RuntimeError(
                "Evaluation task-loop anchor not found."
            )
        source = source.replace(
            task_loop_anchor,
            task_loop_patch,
            1,
        )

    episode_loop_anchor = "    for batch_ix in progbar:\n"
    episode_progress_line = (
        '        print('
        'f"EVAL_PROGRESS '
        "task={_EVAL_PROGRESS['task_index']}/"
        "{_EVAL_PROGRESS['task_total']} "
        'episode={batch_ix + 1}/{n_batches}", '
        "flush=True)\n"
    )
    if "EVAL_PROGRESS task=" not in source:
        if episode_loop_anchor not in source:
            raise RuntimeError(
                "Evaluation episode-loop anchor not found."
            )
        source = source.replace(
            episode_loop_anchor,
            episode_loop_anchor + episode_progress_line,
            1,
        )

    eval_script.write_text(
        source,
        encoding="utf-8",
    )

    libero_plus_path = str(LIBERO_PLUS_DIR)
    sys.path = [
        item
        for item in sys.path
        if item != libero_plus_path
    ]
    sys.path.insert(0, libero_plus_path)

    for module_name in list(sys.modules):
        if (
            module_name == "libero"
            or module_name.startswith("libero.")
            or module_name == "robosuite"
            or module_name.startswith("robosuite.")
        ):
            del sys.modules[module_name]

    importlib.invalidate_caches()

    import libero
    from libero.libero import benchmark

    search_paths = [
        Path(path).resolve()
        for path in getattr(libero, "__path__", [])
    ]

    if not any(
        LIBERO_PLUS_DIR.resolve() in path.parents
        or path == LIBERO_PLUS_DIR.resolve()
        for path in search_paths
    ):
        raise RuntimeError(
            "LIBERO-plus fork was not loaded."
        )

    benchmark_path = Path(
        benchmark.__file__
    ).resolve()

    if (
        LIBERO_PLUS_DIR.resolve()
        not in benchmark_path.parents
    ):
        raise RuntimeError(
            "LIBERO-plus benchmark was not loaded."
        )

    print("LIBERO-plus ready.")


    from libero.libero import benchmark as _holdout_benchmark
    from libero.libero import get_libero_path as _holdout_get_libero_path

    def _holdout_resting_on_static_scene(env, obj_name) -> bool:
        sim = env.sim
        skip = getattr(env, "_support_guard_excluded_geom_ids", None)
        if skip is None:
            skip = set()
            for gid in range(sim.model.ngeom):
                name = sim.model.geom_id2name(gid) or ""
                if name.startswith(("gripper", "robot", "mount")):
                    skip.add(gid)
            for obj in getattr(env, "objects_dict", {}).values():
                for geom in getattr(obj, "contact_geoms", []):
                    try:
                        skip.add(sim.model.geom_name2id(geom))
                    except Exception:
                        pass
            env._support_guard_excluded_geom_ids = skip

        object_geom_ids = set()
        for geom in getattr(env.get_object(obj_name), "contact_geoms", []):
            try:
                object_geom_ids.add(sim.model.geom_name2id(geom))
            except Exception:
                pass
        if not object_geom_ids:
            return True

        for contact_index in range(sim.data.ncon):
            contact = sim.data.contact[contact_index]
            geom1, geom2 = contact.geom1, contact.geom2
            if (
                geom1 in object_geom_ids
                and geom2 not in skip
                and geom2 not in object_geom_ids
            ):
                return True
            if (
                geom2 in object_geom_ids
                and geom1 not in skip
                and geom1 not in object_geom_ids
            ):
                return True
        return False

    # register.py（compe/t1/）が入れている補正パッチと同一のもの。
    # `register.py`自体はimportせず、同じロジックをこの中で完結させている
    # （Colab側にリポジトリ本体が無くても自己完結して動くようにするため）。
    from libero.libero.envs.object_states.base_object_states import SiteObjectState

    if not getattr(SiteObjectState, "_support_contact_guard", False):
        _holdout_original_check_ontop = SiteObjectState.check_ontop

        def _holdout_check_ontop(self, other):
            base = _holdout_original_check_ontop(self, other)
            if not base or self.env.get_object(self.parent_name) is not None:
                return base
            try:
                return _holdout_resting_on_static_scene(self.env, other.object_name)
            except Exception:
                return base

        SiteObjectState.check_ontop = _holdout_check_ontop
        SiteObjectState._support_contact_guard = True

    _HOLDOUT_SUFFIX_RE = re.compile(r"_light_[^.]*|_table_\d+")

    def _holdout_init_states_for(task):
        fn = task.init_states_file
        root = _holdout_get_libero_path("init_states")
        stem, ext = os.path.splitext(fn)
        stripped = _HOLDOUT_SUFFIX_RE.sub("", stem) + ext
        p = os.path.join(root, task.problem_folder, stripped)
        return torch.load(p, weights_only=False)

    _holdout_benchmark.task_maps["libero_holdout"] = {
        row["task_id"]: _holdout_benchmark.Task(
            name=row["task_id"],
            language=row["instruction"],
            problem="Libero",
            problem_folder=row["suite"],
            bddl_file=f"{row['task_id']}.bddl",
            init_states_file=f"{row['task_id']}.pruned_init",
        )
        for row in HOLDOUT_TEST_TASKS
    }
    if "libero_holdout" not in _holdout_benchmark.libero_suites:
        _holdout_benchmark.libero_suites.append("libero_holdout")

    @_holdout_benchmark.register_benchmark
    class LIBERO_HOLDOUT(_holdout_benchmark.Benchmark):
        def __init__(self, task_order_index: int = 0):
            assert task_order_index == 0, (
                "libero_holdout has a variable task count; "
                "only task_order_index=0 supported."
            )
            super().__init__(task_order_index=task_order_index)
            self.name = "libero_holdout"
            self.tasks = list(
                _holdout_benchmark.task_maps[self.name].values()
            )
            self.n_tasks = len(self.tasks)

        def get_task_init_states(self, i):
            return _holdout_init_states_for(self.tasks[i])

    print(
        "libero_holdout suite registered:",
        len(HOLDOUT_TEST_TASKS),
        "tasks",
    )
else:
    print('Skipped LIBERO-plus evaluation setup (MODE=full_submission).')


[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /home/s6323004/PARC2026_pre/.local_libs/parc_lora/venv/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


LIBERO-plus ready.
libero_holdout suite registered: 15 tasks


## 12. holdoutタスクで評価する（`holdout_eval`モードのみ）

`lerobot-eval`ではなく、この競技専用の評価ハーネス`pipeline/`を使う。
実際に`policy_server.py`をHTTPで起動し、`pipeline.rollout.RolloutExecutor`
経由でロールアウトする。これにより：

- 成功判定が「ゴール達成 かつ 非対象物体の変位が閾値（1mm）以下」という
  **本番の採点ロジックと完全に一致**する
- 副産物として、各stepの関節角度・EE位置・action・reward、衝突が
  発生したstep、物体ごとの最大変位量まで手に入る

学習前後の2モデル（ベースライン／LoRA後）を、`libero_holdout`スイート
（15件、L1〜L5 x 3suite）・同じseedで評価する。評価は1モデルにつき
15タスク x `EVAL_EPISODES_PER_TASK`エピソード。関節角度の変化量・EE移動
距離・成功率等は、エピソードごとに移動平均つきでMLflowに記録される。

（`pipeline/rollout.py`には、既存の`object_max_disp`（物体ごとの最大変位）
と衝突タイムステップ・ゴール達成可否をexportするための軽微な追記を行った。
挙動自体は変更していない。）

In [12]:
if MODE == "holdout_eval":
    import subprocess as _eval_subprocess
    import sys as _eval_sys

    import numpy as np

    if str(PARC_REPO_ROOT) not in _eval_sys.path:
        _eval_sys.path.insert(0, str(PARC_REPO_ROOT))

    # このセルから先は、lerobot-evalではなく、この競技専用の評価ハーネス
    # （pipeline/）を使う。実際にpolicy_server.pyをHTTPで起動して叩くので、
    # 「今回の成功判定ロジック（ゴール達成 かつ 非対象物体の変位が閾値以下）」
    # と完全に一致する。副産物として、関節角度・EE位置・衝突タイムステップ等の
    # 生データも手に入る。
    from pipeline.config import EvalConfig, PerturbationConfig
    from pipeline.environment import EnvironmentManager
    from pipeline.remote_policy import RemotePolicyClient
    from pipeline.rollout import RolloutExecutor, TaskResult

    EVAL_MAX_STEPS_PER_EPISODE = 300  # LIBERO系タスクの典型的な上限に合わせた値
    POLICY_SERVER_SCRIPT = (
        PARC_REPO_ROOT / "submission_template" / "policy_server.py"
    )

    eval_pipeline_config = EvalConfig(
        camera_height=256,
        camera_width=256,
        max_steps_per_episode=EVAL_MAX_STEPS_PER_EPISODE,
        n_eval_episodes=EVAL_EPISODES_PER_TASK,
        benchmark_name="libero_holdout",
        seed=EVAL_SEED,
    )
    eval_perturbation = PerturbationConfig()  # 摂動は使わない（holdoutタスク自体が摂動込み）
    eval_env_manager = EnvironmentManager(eval_pipeline_config)
    eval_task_infos = eval_env_manager.get_task_infos("libero_holdout")
    eval_executor = RolloutExecutor(eval_env_manager, eval_pipeline_config)

    _holdout_meta_by_task_id = {
        row["task_id"]: row for row in HOLDOUT_TEST_TASKS
    }


    def _start_policy_server(
        model_dir: Path, port: int
    ) -> _eval_subprocess.Popen:
        server_env = os.environ.copy()
        server_env["SMOLVLA_MODEL_PATH"] = str(model_dir)
        # 標準出力はDEVNULLに捨てる。PIPEにしてどこからも読まないと、
        # uvicornのアクセスログ（1リクエストごとに出る）でパイプの
        # バッファが埋まり、子プロセスがwrite()でブロックして
        # 応答不能になる（実機で実際に再現した）。サーバー側の
        # ログは今回不要なので、素直にDEVNULLへ捨てるのが安全。
        return _eval_subprocess.Popen(
            [sys.executable, "policy_server.py", "--port", str(port)],
            cwd=str(POLICY_SERVER_SCRIPT.parent),
            env=server_env,
            stdout=_eval_subprocess.DEVNULL,
            stderr=_eval_subprocess.DEVNULL,
        )


    def evaluate_model_via_pipeline(
        model_dir: Path, port: int, label: str
    ) -> list[TaskResult]:
        proc = _start_policy_server(model_dir, port)
        client = RemotePolicyClient(
            f"http://localhost:{port}", timeout_sec=15.0
        )
        try:
            client.wait_for_server(timeout=180)
            task_results = []
            for i, task_info in enumerate(eval_task_infos):
                print(
                    f"{label:<13} | task {i + 1}/{len(eval_task_infos)}: "
                    f"{task_info.name}"
                )
                task_result = eval_executor.evaluate_task(
                    client, task_info, eval_perturbation
                )
                task_results.append(task_result)

                # 各stepの生データ（EE位置・関節角度）から補助指標を計算し、
                # 移動平均つきでMLflowに記録する。
                meta = _holdout_meta_by_task_id.get(task_info.name, {})
                tag = f"{meta.get('suite', '?')}_{meta.get('difficulty_level', '?')}"
                for ep in task_result.episodes:
                    ee = np.array(ep.ee_positions) if ep.ee_positions else None
                    ee_travel = (
                        float(
                            np.sum(
                                np.linalg.norm(np.diff(ee, axis=0), axis=1)
                            )
                        )
                        if ee is not None and len(ee) > 1
                        else 0.0
                    )
                    joints = (
                        np.array(ep.joint_positions)
                        if ep.joint_positions
                        else None
                    )
                    joint_delta = (
                        float(
                            np.sum(np.abs(np.diff(joints, axis=0)))
                        )
                        if joints is not None and len(joints) > 1
                        else 0.0
                    )
                    log_metrics_ma(
                        {
                            f"{label}_success": float(ep.success),
                            f"{label}_goal_achieved": float(ep.goal_achieved),
                            f"{label}_collided": float(ep.collided),
                            f"{label}_episode_length": float(ep.total_steps),
                            f"{label}_ee_travel_dist": ee_travel,
                            f"{label}_joint_angle_delta_sum": joint_delta,
                            f"{label}_{tag}_success": float(ep.success),
                        }
                    )
        finally:
            proc.terminate()
            try:
                proc.wait(timeout=15)
            except _eval_subprocess.TimeoutExpired:
                proc.kill()
        return task_results


    def task_results_to_summary(task_results: list) -> dict:
        # 旧eval_info.json（lerobot-eval由来）と互換の最小限のper_task/overall
        # 構造にまとめる（比較セルがこの形を読む）。生データ（joint_positions等）
        # はTaskResult側にそのまま残っているので、必要ならそちらを直接見る。
        per_task = []
        all_successes: list[bool] = []
        for task_result in task_results:
            successes = [ep.success for ep in task_result.episodes]
            per_task.append(
                {
                    "task_id": task_result.task_info.name,
                    "successes": successes,
                    "collided": [ep.collided for ep in task_result.episodes],
                    "goal_achieved": [
                        ep.goal_achieved for ep in task_result.episodes
                    ],
                    "total_steps": [
                        ep.total_steps for ep in task_result.episodes
                    ],
                    "object_max_disp": [
                        ep.object_max_disp for ep in task_result.episodes
                    ],
                    "collision_step": [
                        ep.collision_step for ep in task_result.episodes
                    ],
                }
            )
            all_successes.extend(successes)
        pc_success = (
            100.0 * sum(all_successes) / len(all_successes)
            if all_successes
            else 0.0
        )
        return {"per_task": per_task, "overall": {"pc_success": pc_success}}


    if CACHED_BASE_EVAL_INFO is not None:
        BASE_EVAL_INFO = CACHED_BASE_EVAL_INFO
        print(
            "ベースモデルのholdout評価はキャッシュを使用します"
            "（ベースモデル・holdoutセット・評価パラメータが前回から変わっていません）。"
            " 詳細ロールアウトデータ（関節角度・EE位置等）はキャッシュには含まれない"
            "ため、今回はLoRA後モデル分のみ記録される。"
        )
    else:
        BASE_TASK_RESULTS = evaluate_model_via_pipeline(
            BASELINE_MODEL_DIR, 8931, "base"
        )
        BASE_EVAL_INFO = task_results_to_summary(BASE_TASK_RESULTS)
        save_base_eval_cache(BASE_EVAL_INFO)

    FINETUNED_TASK_RESULTS = evaluate_model_via_pipeline(
        MERGED_MODEL_DIR, 8932, "lora"
    )
    FINETUNED_EVAL_INFO = task_results_to_summary(FINETUNED_TASK_RESULTS)

    print("Evaluation complete.")
else:
    print('Skipped holdout evaluation (MODE=full_submission).')

Base model    | task 1/15 | episode 1/3
Base model    | task 1/15 | episode 2/3
Base model    | task 1/15 | episode 3/3
Base model    | task 2/15 | episode 1/3
Base model    | task 2/15 | episode 2/3
Base model    | task 2/15 | episode 3/3
Base model    | task 3/15 | episode 1/3
Base model    | task 3/15 | episode 2/3
Base model    | task 3/15 | episode 3/3
Base model    | task 4/15 | episode 1/3
Base model    | task 4/15 | episode 2/3
Base model    | task 4/15 | episode 3/3
Base model    | task 5/15 | episode 1/3
Base model    | task 5/15 | episode 2/3
Base model    | task 5/15 | episode 3/3
Base model    | task 6/15 | episode 1/3
Base model    | task 6/15 | episode 2/3
Base model    | task 6/15 | episode 3/3
Base model    | task 7/15 | episode 1/3
Base model    | task 7/15 | episode 2/3
Base model    | task 7/15 | episode 3/3
Base model    | task 8/15 | episode 1/3
Base model    | task 8/15 | episode 2/3
Base model    | task 8/15 | episode 3/3
Base model    | task 9/15 | episode 1/3


## 13. 成功率を比較する（`holdout_eval`モードのみ）

`Δ (pp)`は、追加学習後から追加学習前を引いた成功率差。suite・difficulty・
摂動カテゴリ別の内訳も表示する。

あわせて、摂動ロバスト性の「勾配」も記録する：

- **難易度(L1〜L5)に対する成功率の傾き**（回帰係数）。全体・suiteごとの両方
- **Background TexturesとLight Conditionsの非対称性**（カテゴリ別平均成功率の差）

In [13]:
if MODE == "holdout_eval":
    import numpy as np
    import pandas as pd
    from IPython.display import display


    def per_task_success(eval_info: dict) -> dict[str, float]:
        result: dict[str, float] = {}
        for task_info in eval_info["per_task"]:
            successes = task_info["successes"]
            result[task_info["task_id"]] = (
                100.0 * sum(bool(v) for v in successes) / len(successes)
            )
        return result


    base_per_task = per_task_success(BASE_EVAL_INFO)
    finetuned_per_task = per_task_success(FINETUNED_EVAL_INFO)

    _difficulty_to_int = {"L1": 1, "L2": 2, "L3": 3, "L4": 4, "L5": 5}

    rows = []

    for meta in HOLDOUT_TEST_TASKS:
        task_id = meta["task_id"]
        base_score = base_per_task[task_id]
        finetuned_score = finetuned_per_task[task_id]

        rows.append(
            {
                "Suite": meta["suite"],
                "Difficulty": meta["difficulty_level"],
                "Category": meta["category"],
                "Instruction": meta["instruction"],
                "Base (%)": base_score,
                "PARC LoRA (%)": finetuned_score,
                "Δ (pp)": finetuned_score - base_score,
            }
        )

    base_overall = float(BASE_EVAL_INFO["overall"]["pc_success"])
    finetuned_overall = float(FINETUNED_EVAL_INFO["overall"]["pc_success"])

    detail_df = pd.DataFrame(rows)

    rows.append(
        {
            "Suite": "Overall",
            "Difficulty": "",
            "Category": "",
            "Instruction": "(all 15 holdout test cases)",
            "Base (%)": base_overall,
            "PARC LoRA (%)": finetuned_overall,
            "Δ (pp)": finetuned_overall - base_overall,
        }
    )

    comparison_df = pd.DataFrame(rows)
    comparison_df.to_csv(COMPARISON_CSV_PATH, index=False)

    display(comparison_df.round(1))

    print(
        f"Overall: {base_overall:.1f}% → "
        f"{finetuned_overall:.1f}% "
        f"({finetuned_overall - base_overall:+.1f} pp)"
    )

    mlflow.log_metrics(
        {
            "holdout_base_pc_success": base_overall,
            "holdout_finetuned_pc_success": finetuned_overall,
            "holdout_delta_pp": finetuned_overall - base_overall,
        }
    )
    for row in rows:
        if row["Suite"] == "Overall":
            continue
        tag = f"{row['Suite']}_{row['Difficulty']}"
        mlflow.log_metric(f"holdout_delta_pp_{tag}", row["Δ (pp)"])
    mlflow.log_artifact(str(COMPARISON_CSV_PATH))

    # --- 項目9: 摂動ロバスト性の「勾配」 ---
    # (a) 難易度レベル(L1〜L5)に対する成功率の傾き（単回帰の係数）。
    #     全15点をプールした全体の傾きに加え、suiteごとの傾きも見る。
    #     値が大きく負であるほど、難易度が上がるにつれ性能が急落することを示す。
    def _slope(difficulties: list[int], scores: list[float]) -> float:
        if len(set(difficulties)) < 2:
            return float("nan")
        x = np.asarray(difficulties, dtype=float)
        y = np.asarray(scores, dtype=float)
        slope, _intercept = np.polyfit(x, y, 1)
        return float(slope)

    for model_label, per_task in (
        ("base", base_per_task),
        ("finetuned", finetuned_per_task),
    ):
        overall_diff = [
            _difficulty_to_int[m["difficulty_level"]] for m in HOLDOUT_TEST_TASKS
        ]
        overall_scores = [per_task[m["task_id"]] for m in HOLDOUT_TEST_TASKS]
        mlflow.log_metric(
            f"holdout_{model_label}_difficulty_slope",
            _slope(overall_diff, overall_scores),
        )
        for suite in SUITE_TASK_NAMES:
            suite_diff = [
                _difficulty_to_int[m["difficulty_level"]]
                for m in HOLDOUT_TEST_TASKS
                if m["suite"] == suite
            ]
            suite_scores = [
                per_task[m["task_id"]]
                for m in HOLDOUT_TEST_TASKS
                if m["suite"] == suite
            ]
            mlflow.log_metric(
                f"holdout_{model_label}_difficulty_slope_{suite}",
                _slope(suite_diff, suite_scores),
            )

    # (b) Background Textures と Light Conditions のロバスト性の非対称性
    #     （カテゴリ別平均成功率の差。正なら明暗より背景に強い、負ならその逆）。
    for model_label, per_task in (
        ("base", base_per_task),
        ("finetuned", finetuned_per_task),
    ):
        by_category: dict[str, list[float]] = {}
        for m in HOLDOUT_TEST_TASKS:
            by_category.setdefault(m["category"], []).append(
                per_task[m["task_id"]]
            )
        category_means = {
            cat: float(np.mean(scores)) for cat, scores in by_category.items()
        }
        for cat, mean_score in category_means.items():
            mlflow.log_metric(
                f"holdout_{model_label}_category_mean_{cat.replace(' ', '_')}",
                mean_score,
            )
        if "Background Textures" in category_means and "Light Conditions" in category_means:
            asymmetry = (
                category_means["Background Textures"]
                - category_means["Light Conditions"]
            )
            mlflow.log_metric(
                f"holdout_{model_label}_bg_vs_light_asymmetry_pp",
                asymmetry,
            )

    print("項目9（difficulty slope・カテゴリ非対称性）をMLflowに記録しました。")
else:
    print('Skipped comparison (MODE=full_submission).')

,Suite,Difficulty,Category,Instruction,Base (%),PARC LoRA (%),Δ (pp)
0,libero_spatial,L1,Background Textures,Pick the akita black bowl in the top layer of ...,33.3,33.3,0.0
1,libero_spatial,L2,Background Textures,Pick the akita black bowl in the top layer of ...,0.0,0.0,0.0
2,libero_spatial,L3,Background Textures,Pick the akita black bowl in the top layer of ...,0.0,33.3,33.3
3,libero_spatial,L4,Background Textures,Pick the akita black bowl in the top layer of ...,66.7,33.3,-33.3
4,libero_spatial,L5,Light Conditions,Pick the akita black bowl in the top layer of ...,0.0,0.0,0.0
5,libero_object,L1,Background Textures,Pick the bbq sauce and place it in the basket,66.7,66.7,0.0
6,libero_object,L2,Background Textures,Pick the bbq sauce and place it in the basket,33.3,100.0,66.7
7,libero_object,L3,Background Textures,Pick the bbq sauce and place it in the basket,0.0,0.0,0.0
8,libero_object,L4,Background Textures,Pick the bbq sauce and place it in the basket,33.3,33.3,0.0
9,libero_object,L5,Background Textures,Pick the bbq sauce and place it in the basket,0.0,0.0,0.0


Overall: 42.2% → 46.7% (+4.4 pp)


## 14. モデル（と比較結果）をダウンロードする

マージ済みモデルは常にzipでダウンロードする。`holdout_eval`モードでは
比較結果CSVも合わせてダウンロードする。

`full_submission`モードでダウンロードしたモデルを提出物に組み込む手順
（このNotebookの外で行う）:

1. ダウンロードしたzipを展開し、`submission_template/model_weights/`配下
   などローカルに配置する。
2. `submission_template/policy_server.py`の`SMOLVLA_MODEL_PATH`環境変数
   （またはデフォルト値）を、展開したモデルディレクトリに向ける。
3. `src/download_model_weights.py`と同様の要領で、必要ならHFキャッシュ
   同梱に切り替える（`my_strategy.md`方針4参照）。
4. `validate_submission.py`で静的・動的チェックを通してから提出zipを作る。

In [14]:
from zipfile import ZIP_STORED, ZipFile

try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if MERGED_ZIP_PATH.exists():
    MERGED_ZIP_PATH.unlink()

with ZipFile(
    MERGED_ZIP_PATH,
    mode="w",
    compression=ZIP_STORED,
    allowZip64=True,
) as archive:
    for file_path in sorted(
        MERGED_MODEL_DIR.rglob("*")
    ):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=(
                    Path(MERGED_MODEL_DIR.name)
                    / file_path.relative_to(
                        MERGED_MODEL_DIR
                    )
                ),
            )

print(f"Saved: {MERGED_ZIP_PATH}")

if _IN_COLAB:
    files.download(str(MERGED_ZIP_PATH))
    if MODE == "holdout_eval" and COMPARISON_CSV_PATH.exists():
        files.download(str(COMPARISON_CSV_PATH))
else:
    print("Colab外での実行のため、ダウンロードはスキップした。")
    print(f"モデル: {MERGED_ZIP_PATH}")
    if MODE == "holdout_eval" and COMPARISON_CSV_PATH.exists():
        print(f"比較結果: {COMPARISON_CSV_PATH}")

if MODE == "full_submission":
    print()
    print(
        "次のステップ: このzipをsubmission_template/model_weights/配下に "
        "展開し、SMOLVLA_MODEL_PATHをそのディレクトリに向けたうえで "
        "validate_submission.pyを実行してください。"
    )

mlflow.set_tag("merged_zip_path", str(MERGED_ZIP_PATH))
mlflow.end_run()
print(f"MLflow run終了: {MLFLOW_RUN.info.run_id}")

Saved: /home/s6323004/parc_lora_workspace/smolvla_parc_lora_holdout_eval_merged.zip
Colab外での実行のため、ダウンロードはスキップした。
モデル: /home/s6323004/parc_lora_workspace/smolvla_parc_lora_holdout_eval_merged.zip
比較結果: /home/s6323004/parc_lora_workspace/parc_holdout_comparison_holdout_eval.csv


## 15. 提出物を組み立てて`validate_submission.py`で検証する（`full_submission`モードのみ）

`full_submission`モードで学習したマージ済みモデルを使って、実際に提出する形（
`submission_template/`と同じ構成 + マージ済みモデル）の提出物一式を組み立て、
このリポジトリの`validate_submission.py`（静的チェック + 実際にサーバーを起動
する動的スモークテスト）を自動で実行する。

- リポジトリがローカルに無い場合（Colabなど）はスキップする。
- `holdout_eval`モードでは意図的に学習データを間引いているため、提出物としては
  扱わずスキップする（`MODE`を`full_submission`にしてから実行すること）。

In [15]:
if MODE != "full_submission":
    print("Skipped submission packaging (MODE=holdout_eval).")
elif "PARC_REPO_ROOT" not in dir():
    print(
        "Skipped submission packaging: リポジトリがローカルに見つからない"
        "（Colab等、PARC_REPO_ROOTが未定義）。"
    )
else:
    import re
    import shutil
    import subprocess as _subprocess

    SUBMISSION_TEMPLATE_SRC = PARC_REPO_ROOT / "submission_template"
    VALIDATE_SCRIPT = PARC_REPO_ROOT / "validate_submission.py"

    if not SUBMISSION_TEMPLATE_SRC.is_dir() or not VALIDATE_SCRIPT.is_file():
        print(
            "Skipped submission packaging: "
            f"{SUBMISSION_TEMPLATE_SRC} または {VALIDATE_SCRIPT} が見つからない。"
        )
    else:
        PACKAGE_DIR = WORKSPACE_ROOT / "submission_package"
        SUBMISSION_ZIP_PATH = WORKSPACE_ROOT / "submission.zip"
        MERGED_MODEL_SUBDIR = "lora_merged"

        shutil.rmtree(PACKAGE_DIR, ignore_errors=True)
        PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

        # submission_template/ から policy_server.py 等を持ってくる
        # （既存の model_weights/ はベース重み用なので持ち込まず、後で
        # マージ済みモデルだけを新規に配置する）。
        for item in SUBMISSION_TEMPLATE_SRC.iterdir():
            if item.name == "model_weights":
                continue
            dest = PACKAGE_DIR / item.name
            if item.is_dir():
                shutil.copytree(item, dest)
            else:
                shutil.copy2(item, dest)

        # マージ済みモデルを配置する
        merged_dest = PACKAGE_DIR / "model_weights" / MERGED_MODEL_SUBDIR
        merged_dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(MERGED_MODEL_DIR, merged_dest)

        # policy_server.py の SMOLVLA_MODEL_PATH デフォルト値を、同梱した
        # マージ済みモデルに向ける（本番の採点環境は環境変数を設定してくれ
        # ないため、コード側のデフォルトを書き換える必要がある。__file__基準の
        # 絶対パスにすることで、どのカレントディレクトリから起動されても
        # 正しく解決できるようにする）。
        packaged_server = PACKAGE_DIR / "policy_server.py"
        server_src = packaged_server.read_text(encoding="utf-8")
        old_line = (
            'MODEL_PATH = os.environ.get("SMOLVLA_MODEL_PATH", '
            '"lerobot/smolvla_libero_plus")'
        )
        new_line = (
            "MODEL_PATH = os.environ.get(\n"
            '        "SMOLVLA_MODEL_PATH",\n'
            "        str(\n"
            "            pathlib.Path(__file__).resolve().parent\n"
            f'            / "model_weights" / "{MERGED_MODEL_SUBDIR}"\n'
            "        ),\n"
            "    )"
        )
        if old_line not in server_src:
            raise RuntimeError(
                "policy_server.py 内の MODEL_PATH 行が見つからず、"
                "書き換えできなかった。"
            )
        server_src = server_src.replace(old_line, new_line, 1)
        packaged_server.write_text(server_src, encoding="utf-8")

        # 中身を直接zip化する（policy_server.pyがzipルート直下に来るように）。
        if SUBMISSION_ZIP_PATH.exists():
            SUBMISSION_ZIP_PATH.unlink()
        _subprocess.run(
            ["zip", "-rq", "-X", str(SUBMISSION_ZIP_PATH), "."],
            cwd=str(PACKAGE_DIR),
            check=True,
        )
        print(f"Submission zip: {SUBMISSION_ZIP_PATH}")

        # validate_submission.py --install を実行する（実際にサーバーを起動する
        # 動的スモークテストまで含む）。
        print("Running validate_submission.py --install ...")
        result = _subprocess.run(
            [
                sys.executable,
                str(VALIDATE_SCRIPT),
                "--install",
                str(SUBMISSION_ZIP_PATH),
            ],
            cwd=str(PARC_REPO_ROOT),
            capture_output=True,
            text=True,
        )
        print(result.stdout)
        if result.stderr.strip():
            print(result.stderr)

        if result.returncode != 0:
            raise RuntimeError(
                "validate_submission.py がFAILを返した。"
                "上の出力を確認して修正すること。"
            )

        print(f"検証済み。提出用zip: {SUBMISSION_ZIP_PATH}")

Skipped submission packaging (MODE=holdout_eval).
